In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier


In [2]:
# 1. Automatically find and switch to the project root directory
current_path = Path(os.getcwd()).resolve()
root_name = "student_engagement_twin"
root_dir = None

for p in [current_path] + list(current_path.parents):
    if p.name == root_name:
        root_dir = p
        break

if root_dir is None:
    root_dir = current_path
    print(f"⚠️ Project root folder not found in parents. Using: {root_dir}")
else:
    os.chdir(root_dir)
    print(f" Switched to project root: {root_dir}")

 Switched to project root: D:\student_engagement_twin


In [3]:
# Ensure the 'src' directory can be discovered
sys.path.append(str(root_dir / "src"))

In [4]:
# 2. Check for processed files and load them
PROCESSED_DIR = Path("data/processed")
FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
TARGET = "Engagement_Level"
SEED = 42

if not (PROCESSED_DIR / "train.csv").exists():
    raise FileNotFoundError(
        f"train.csv not found at {PROCESSED_DIR.resolve() / 'train.csv'}. "
        "Please run your data preparation code or notebook cells first!"
    )

train = pd.read_csv(PROCESSED_DIR / "train.csv")
print(f" Loaded train dataset successfully. Shape: {train.shape}")

 Loaded train dataset successfully. Shape: (21328, 5)


In [5]:
# 3. Setup features and target matrices
X, y = train[FEATURES], train[TARGET]

In [6]:
# 4. Initialize and run your cross-validated XGBoost model
labeler = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=SEED,
    eval_metric="logloss",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pred = cross_val_predict(labeler, X, y, cv=cv)

In [7]:
# 5. Evaluate the model performance
print("\nClassification Report:")
print(classification_report(y, pred, digits=3))
print("\nConfusion Matrix:")
print(confusion_matrix(y, pred))


Classification Report:
              precision    recall  f1-score   support

           0      0.932     0.953     0.942     17587
           1      0.753     0.672     0.710      3741

    accuracy                          0.904     21328
   macro avg      0.843     0.812     0.826     21328
weighted avg      0.900     0.904     0.902     21328


Confusion Matrix:
[[16764   823]
 [ 1228  2513]]


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

SEED = 42

labeler = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=SEED,
    eval_metric="logloss",
)   # XGBoost handles NaN natively, so we keep the NaNs

X, y = train[FEATURES], train[TARGET]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pred = cross_val_predict(labeler, X, y, cv=cv)

print(classification_report(y, pred, digits=3))
print(confusion_matrix(y, pred))

              precision    recall  f1-score   support

           0      0.932     0.953     0.942     17587
           1      0.753     0.672     0.710      3741

    accuracy                          0.904     21328
   macro avg      0.843     0.812     0.826     21328
weighted avg      0.900     0.904     0.902     21328

[[16764   823]
 [ 1228  2513]]


Locate the project root

In [10]:
from pathlib import Path


def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

Project root: D:\student_engagement_twin


 Fit the final labeler and save it

train the labeler on the *entire* real training set and save it to disk.

In [11]:
import joblib

labeler.fit(X, y)

artifacts_dir = PROJECT_ROOT / "artifacts"
artifacts_dir.mkdir(exist_ok=True)
joblib.dump(labeler, artifacts_dir / "labeler.joblib")

share = labeler.predict(X).mean()
print("Saved:", artifacts_dir / "labeler.joblib")
print(f"Predicted class-1 share on train: {share:.4f}  (real: {y.mean():.4f})")

Saved: D:\student_engagement_twin\artifacts\labeler.joblib
Predicted class-1 share on train: 0.1567  (real: 0.1754)
